# Expérience 4 — SVD (bibliothèque Surprise)

Surprise attend des **notes explicites**, absentes du jeu de données. Trois
définitions de note ont été implémentées ; ce notebook les compare sur le même
protocole que les autres méthodes.

| Définition | Construction | Notebook de référence |
|---|---|---|
| étoiles de l'article | clics reçus par l'article, échelle log 1–5 | `02_notation_etoiles.ipynb` |
| intensité par couple | nombre de clics du couple (lecteur, article) | — |
| binaire + négatifs | lu = 1, non lu échantillonné = 0 | — |

⚠️ Chaque entraînement prend plusieurs minutes. Les trois variantes sont
entraînées par le notebook lui-même, sur la **seule** période d'entraînement : on
ne se sert pas des artefacts de `models_split/`, dont la variante peut changer
d'une reconstruction à l'autre.

## Protocole commun

Identique dans tous les notebooks d'expérimentation, sinon les chiffres ne sont pas
comparables :

- **découpage temporel 60 / 20 / 20** sur `click_timestamp` ;
- artefacts construits sur la **seule** période d'entraînement (`models_split/`) ;
- réglage sur la **validation** ; la période de test reste intacte jusqu'à la mesure
  finale (notebook 07) ;
- lecteurs évalués : connus à l'entraînement **et** actifs pendant la période
  d'évaluation ;
- métriques : HitRate@5, Recall@5, couverture, personnalisation.

> Prérequis : `python -m src.evaluate --data-dir data/news-portal-user --out-dir models_split`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src import experiments as xp
from src.recommender import Recommender

DATA = Path('..') / 'data' / 'news-portal-user'
MODELS = Path('..') / 'models_split'

train, val, test = xp.load_split(DATA)
reco = Recommender(MODELS)
users, cible = xp.eval_users(reco, val, max_users=2000)
print(f'{len(users):,} lecteurs évalués sur la période de validation')

[clicks] 1 fichier(s) vide(s) ignoré(s) : clicks_hour_100.csv


[split] entraînement 1,792,908 | validation 597,636 | test 597,637
[split] bornes temporelles : t60=1507602953792 t80=1507843212579


2,000 lecteurs évalués sur la période de validation


## 1. Variante « étoiles » (artefacts existants)

Comparée à la popularité récente, qui sert de référence.

In [2]:
POOL_HEURES = 6
pool = xp.recent_pool(train, POOL_HEURES)

# La variante « étoiles » est entraînée ici, et non lue dans `models_split/` :
# l'étiquette d'une ligne doit dire ce qui a été mesuré, sans dépendre de ce que
# le dossier d'artefacts contient au moment du calcul. Cette ligne a déjà affiché
# les chiffres d'une autre variante après une reconstruction des artefacts, sans
# qu'aucune erreur ne se lève.
from src.evaluate import svd_strategie

configs = {
    'SVD étoiles': svd_strategie(reco, train, pool, negatives=0,
                                 models_dir=MODELS),
    'ALS (référence)': xp.make_als(reco, pool),
    'popularité récente (référence)': xp.make_popularity(pool, reco),
}
xp.compare(configs, users, cible, n_articles=reco.n_articles)

[notes] 1,769,009 couples notés à partir des étoiles — 1★:42,080 · 2★:131,166 · 3★:327,661 · 4★:912,849 · 5★:355,253


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
SVD étoiles,0.0015,0.0000,0.005,22.9
ALS (référence),0.0120,0.0016,0.031,84.3
popularité récente (référence),0.1480,0.0246,0.004,7.7


## 2. Variante binaire avec échantillonnage négatif

Une note qui ne dépend que de l'article ne peut pas personnaliser : le biais
article capte tout le signal. L'alternative est de poser lu = 1 / non lu = 0, ce qui
transforme la prédiction de note en tâche de classement.

Le nombre de négatifs par positif est le réglage à explorer.

In [3]:
# `svd_strategie` entraîne la variante demandée en mémoire : plus de dossier
# temporaire de 60 Mo, et aucune dépendance à l'ordre des écritures sur disque.
configs = {'popularité récente (référence)': xp.make_popularity(pool, reco)}
for negatifs in (1, 4):
    configs[f'SVD binaire, {negatifs} négatif(s)'] = svd_strategie(
        reco, train, pool, negatives=negatifs)

xp.compare(configs, users, cible, n_articles=reco.n_articles)

[notes] 1,769,009 positifs + 1,767,138 négatifs = 3,536,147 exemples


[notes] 1,769,009 positifs + 7,061,782 négatifs = 8,830,791 exemples


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
popularité récente (référence),0.1480,0.0246,0.004,7.7
"SVD binaire, 1 négatif(s)",0.0080,0.0013,0.037,46.1
"SVD binaire, 4 négatif(s)",0.0815,0.0119,0.013,34.3


## Lecture

Trois définitions de note, trois résultats très différents — **avec la même
bibliothèque et le même algorithme**, mesurés sur la validation :

| Définition de la note | HitRate@5 | Personnalisation |
|---|---|---|
| étoiles de l'article | 0,0015 | 22,9 % |
| binaire, 1 négatif | 0,0080 | 46,1 % |
| binaire, 4 négatifs | **0,0815** | 34,3 % |

Référence sur la même mesure : popularité récente 6 h, 0,1480.

**Ce qui compte n'est pas la bibliothèque mais la formulation du problème.**

Les étoiles échouent parce que la note ne dépend que de l'article : deux lecteurs
attribuent la même valeur au même article, le biais article capte donc tout le
signal et le modèle apprend « quels articles sont lus », pas « qui lit quoi ». Il
personnalise (22,9 %) sans jamais tomber juste — le contraire de ce qu'on
attend d'un classement.

Les notes binaires avec négatifs réparent cela : en donnant au modèle des exemples
« non lu = 0 », on transforme une prédiction de note en tâche de **classement**,
qui est la vraie question posée. Le gain est d'un facteur 54.

### Réserve importante sur ces chiffres

Le 0,0815 de la variante à 4 négatifs **ne s'est pas reproduit sur la période de
test** : elle y obtient 0,0220 (voir notebook 07), soit 3,7 fois moins. Elle reste
devant le contenu (0,0200) mais passe derrière l'ALS (0,0415), et à deux ordres de
grandeur de la popularité récente (0,2525).

Ce réglage est donc **instable** : il dépend fortement du vivier de candidats et de
la période. On ne peut pas conclure que le SVD dépasse l'ALS — seulement que la
définition de la note change tout, ce qui reste vrai dans les deux mesures.

C'est aussi la raison d'être de la séparation validation / test : sans elle, nous
aurions annoncé 0,0815 comme un résultat.

### Deux pièges rencontrés ici, et corrigés

1. **La ligne « étoiles » lisait les artefacts du dossier** au lieu d'entraîner la
   variante qu'elle annonçait. Après une reconstruction de `models_split/` avec une
   autre variante, elle a affiché des chiffres qui n'étaient pas les siens, sans
   qu'aucune erreur ne se lève. Les trois variantes sont désormais entraînées par
   le notebook, via `src.evaluate.svd_strategie`.
2. **Le SVD était le seul modèle à ne pas être ré-entraîné** sur l'historique de la
   mesure, alors que l'ALS l'était. Sur le test il tombait à 0,0005 — non par
   faiblesse, mais parce qu'il ignorait la période de validation. Comparaison
   inégale, corrigée dans `src/evaluate.py`.